# FER2013 Enhanced Emotion Classification

Notebook nay dung tren Google Colab de train model CNN phan loai cam xuc khuon mat.

Pipeline:
1. Tai dataset `abhilash88/fer2013-enhanced`
2. Khao sat du lieu
3. Tien xu ly anh
4. Tao data pipeline
5. Xay dung CNN
6. Huan luyen
7. Danh gia
8. Luu model va label mapping
9. Tai ket qua ve project local


## 0. Cai thu vien

Colab thuong da co TensorFlow. O buoc dau tien chi can cai cac thu vien con lai dung cho dataset va bao cao.

In [ ]:
!pip -q install datasets scikit-learn matplotlib seaborn pillow

## 1. Kiem tra moi truong

In [ ]:
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
from datasets import load_dataset

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

Path("models").mkdir(exist_ok=True)
Path("reports").mkdir(exist_ok=True)
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

## 2. Tai dataset fer2013-enhanced

Day la step dau tien cua pipeline. Dataset se duoc cache trong moi truong Colab, khong can tai ve may local.

In [ ]:
DATASET_NAME = "abhilash88/fer2013-enhanced"
EXPECTED_SPLITS = ("train", "validation", "test")
IMAGE_COLUMNS = ("image", "pixels")
LABEL_COLUMNS = ("emotion", "emotion_name")

dataset = load_dataset(DATASET_NAME)
dataset

## 3. Xac nhan split va cot du lieu

In [ ]:
def find_first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


summary = {
    "dataset_name": DATASET_NAME,
    "splits": {},
}

for split_name, split_data in dataset.items():
    columns = list(split_data.column_names)
    image_column = find_first_existing(columns, IMAGE_COLUMNS)
    label_column = find_first_existing(columns, LABEL_COLUMNS)
    summary["splits"][split_name] = {
        "num_rows": len(split_data),
        "columns": columns,
        "image_column": image_column,
        "label_column": label_column,
    }

missing_splits = [split for split in EXPECTED_SPLITS if split not in summary["splits"]]
if missing_splits:
    raise ValueError(f"Missing expected split(s): {missing_splits}")

for split_name, split_summary in summary["splits"].items():
    if split_summary["image_column"] is None:
        raise ValueError(f"Split '{split_name}' has no image column. Columns: {split_summary['columns']}")
    if split_summary["label_column"] is None:
        raise ValueError(f"Split '{split_name}' has no label column. Columns: {split_summary['columns']}")

summary_path = Path("reports/dataset_load_summary.json")
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Loaded dataset: {DATASET_NAME}")
for split_name in EXPECTED_SPLITS:
    split_summary = summary["splits"][split_name]
    print(
        f"- {split_name}: {split_summary['num_rows']} rows, "
        f"image='{split_summary['image_column']}', label='{split_summary['label_column']}'"
    )
print(f"Saved summary to: {summary_path}")

## 4. Xem thu mot mau du lieu

Cell nay giup xac nhan dataset co dang anh/nhan dung nhu mong doi. Buoc khao sat chi tiet se lam o step 2.

In [ ]:
sample = dataset["train"][0]
print(sample.keys())

image_column = summary["splits"]["train"]["image_column"]
label_column = summary["splits"]["train"]["label_column"]

print("Label:", sample[label_column])
sample[image_column]

## 5. Step 2 - Khao sat du lieu

Muc tieu:
- Dem so anh moi split.
- Dem so anh moi class.
- Xac nhan label mapping.
- Kiem tra kich thuoc anh.
- Xem anh mau theo class.
- Khao sat cac cot chat luong anh neu co.

In [ ]:
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style="whitegrid")

CANONICAL_LABELS = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "sad",
    "surprise",
    "neutral",
]

def normalize_name(name):
    return str(name).strip().lower().replace(" ", "_")

### 5.1. Xac dinh label mapping

In [ ]:
label_mapping = {}

for split_name in EXPECTED_SPLITS:
    split_data = dataset[split_name]
    for row in split_data:
        label_id = int(row["emotion"])
        label_name = normalize_name(row.get("emotion_name", label_id))
        label_mapping[label_id] = label_name
    break

label_mapping = dict(sorted(label_mapping.items()))
print(label_mapping)

Path("models").mkdir(exist_ok=True)
Path("models/labels.json").write_text(json.dumps({str(k): v for k, v in label_mapping.items()}, indent=2), encoding="utf-8")
print("Saved label mapping to: models/labels.json")

### 5.2. Dem so luong class theo split

In [ ]:
class_counts = {}

for split_name in EXPECTED_SPLITS:
    counts = Counter(int(row["emotion"]) for row in dataset[split_name])
    class_counts[split_name] = {
        label_mapping[label_id]: counts.get(label_id, 0)
        for label_id in sorted(label_mapping)
    }

print(json.dumps(class_counts, indent=2))

Path("reports/class_distribution.json").write_text(json.dumps(class_counts, indent=2), encoding="utf-8")
print("Saved class counts to: reports/class_distribution.json")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, split_name in zip(axes, EXPECTED_SPLITS):
    labels = list(class_counts[split_name].keys())
    values = list(class_counts[split_name].values())
    sns.barplot(x=labels, y=values, ax=ax, color="#4c78a8")
    ax.set_title(split_name)
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.savefig("reports/class_distribution.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved chart to: reports/class_distribution.png")

### 5.3. Kiem tra kich thuoc anh va dinh dang anh

In [ ]:
def to_pil_image(value):
    if isinstance(value, Image.Image):
        return value

    array = np.array(value)
    if array.ndim == 1:
        side = int(np.sqrt(array.size))
        array = array.reshape(side, side)
    return Image.fromarray(array.astype("uint8"))


image_stats = {}

for split_name in EXPECTED_SPLITS:
    sizes = Counter()
    modes = Counter()
    for row in dataset[split_name].select(range(min(500, len(dataset[split_name])))):
        image = to_pil_image(row["image"])
        sizes[image.size] += 1
        modes[image.mode] += 1
    image_stats[split_name] = {
        "sizes": {str(k): v for k, v in sizes.items()},
        "modes": dict(modes),
    }

print(json.dumps(image_stats, indent=2))
Path("reports/image_stats.json").write_text(json.dumps(image_stats, indent=2), encoding="utf-8")
print("Saved image stats to: reports/image_stats.json")

### 5.4. Xem anh mau theo tung class

In [ ]:
examples_by_class = {}

for row in dataset["train"]:
    label_id = int(row["emotion"])
    if label_id not in examples_by_class:
        examples_by_class[label_id] = row
    if len(examples_by_class) == len(label_mapping):
        break

fig, axes = plt.subplots(1, len(label_mapping), figsize=(15, 3))
for ax, label_id in zip(axes, sorted(label_mapping)):
    row = examples_by_class[label_id]
    image = to_pil_image(row["image"])
    ax.imshow(image, cmap="gray")
    ax.set_title(label_mapping[label_id])
    ax.axis("off")

plt.tight_layout()
plt.savefig("reports/sample_images_by_class.png", dpi=160, bbox_inches="tight")
plt.show()
print("Saved sample images to: reports/sample_images_by_class.png")

### 5.5. Khao sat cac chi so chat luong anh

In [ ]:
quality_columns = [
    column
    for column in ["quality_score", "brightness", "contrast", "focus_score", "brightness_score"]
    if column in dataset["train"].column_names
]

quality_summary = defaultdict(dict)

for split_name in EXPECTED_SPLITS:
    for column in quality_columns:
        values = np.array([row[column] for row in dataset[split_name]], dtype="float32")
        quality_summary[split_name][column] = {
            "min": float(np.min(values)),
            "mean": float(np.mean(values)),
            "median": float(np.median(values)),
            "max": float(np.max(values)),
        }

print(json.dumps(quality_summary, indent=2))
Path("reports/quality_summary.json").write_text(json.dumps(quality_summary, indent=2), encoding="utf-8")
print("Saved quality summary to: reports/quality_summary.json")

In [ ]:
if quality_columns:
    fig, axes = plt.subplots(len(quality_columns), 1, figsize=(10, 3 * len(quality_columns)))
    if len(quality_columns) == 1:
        axes = [axes]

    for ax, column in zip(axes, quality_columns):
        values = np.array([row[column] for row in dataset["train"]], dtype="float32")
        sns.histplot(values, bins=40, ax=ax, color="#59a14f")
        ax.set_title(f"Train {column}")
        ax.set_xlabel(column)

    plt.tight_layout()
    plt.savefig("reports/quality_histograms.png", dpi=160, bbox_inches="tight")
    plt.show()
    print("Saved quality histograms to: reports/quality_histograms.png")
else:
    print("No quality columns found.")